# Heterogeneous GraphSAGE on MLCommons / MLPerf Results

This notebook reads exactly two checked-in exports from `data/raw`, with a
fallback to `~/Downloads/mlcommons_results_latest`:

- `Table - Inference_Full Data_data.csv` (Datacenter v6.1)
- `Table - Inference_edge.csv` (Edge v6.1)

normalizes the results into one long table, builds a heterogeneous
system–hardware–workload graph, and trains GraphSAGE with system-grouped
validation.

The output is a performance-prediction experiment. It is **not** an official
MLPerf result or a reproduction of the MLPerf benchmark. The summary CSVs do
not contain general accuracy evidence or hardware profiler counters.

In [ ]:
from pathlib import Path
from types import SimpleNamespace
import os
import sys

cache_dir = Path.cwd() / ".cache"
cache_dir.mkdir(exist_ok=True)
os.environ.setdefault("MPLCONFIGDIR", str(cache_dir / "matplotlib"))
os.environ.setdefault("XDG_CACHE_HOME", str(cache_dir))

import matplotlib
from IPython import get_ipython
if get_ipython() is not None:
    get_ipython().run_line_magic("matplotlib", "inline")
else:
    matplotlib.use("Agg")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from IPython.display import display

PROJECT_ROOT = Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT / "src"))
from ai_benchmark import prepare_data, train_graphsage as gnn

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 160)

print("native machine architecture:", __import__("platform").machine())
print("torch:", torch.__version__)
print("model module:", gnn.__file__)
print("CUDA available:", torch.cuda.is_available())

## 1. Load the Datacenter and Edge CSV exports

Only these two files are accepted. They are not the same export or the same
content, so the notebook verifies their SHA-256 hashes and stops if they are
byte-identical. The files are UTF-16, tab-separated CSV exports.

In [ ]:
DOWNLOADS = Path.home() / "Downloads"
LOCAL_RESULTS = PROJECT_ROOT / "data" / "raw"
RESULT_FILES = [
    "Table - Inference_Full Data_data.csv",
    "Table - Inference_edge.csv",
]


def find_results_dir(downloads: Path, filenames: list[str]) -> Path:
    preferred = downloads / "mlcommons_results_latest"
    candidates = [LOCAL_RESULTS, preferred, downloads]
    candidates.extend(
        sorted(
            path
            for path in downloads.iterdir()
            if path.is_dir()
            and ("mlperf" in path.name.lower() or "mlcommons" in path.name.lower())
        )
    )
    for candidate in candidates:
        if all((candidate / filename).exists() for filename in filenames):
            return candidate
    raise FileNotFoundError(
        "Could not find the results export under Downloads. "
        "Set RESULTS_DIR manually to the folder containing the two expected CSV files."
    )


RESULTS_DIR = find_results_dir(DOWNLOADS, RESULT_FILES)
integrity = prepare_data.source_integrity(RESULTS_DIR)
assert integrity["sha256"].nunique() == 2
print("Using:", RESULTS_DIR)
display(integrity)

In [ ]:
long_table = prepare_data.build_dataset(RESULTS_DIR)
canonical_path = Path("data/mlperf_long.csv")
canonical_path.parent.mkdir(parents=True, exist_ok=True)
long_table.to_csv(canonical_path, index=False)
long_table = pd.read_csv(canonical_path)

audit = (
    long_table.groupby(["suite", "version", "division"], dropna=False)
    .agg(
        rows=("result", "size"),
        systems=("system_id", "nunique"),
        submissions=("public_id", "nunique"),
        target_groups=("target_group", "nunique"),
    )
    .reset_index()
)
print(f"Wrote {len(long_table):,} normalized rows to {canonical_path.resolve()}")
display(audit)

## 2. Configure the experiment

`SUITE="all"` uses both Datacenter and Edge. Their target groups remain
separate through suite, version, division, model, task, scenario, and unit.
Set `SUITE="datacenter"` or `"edge"` to evaluate one table at a time.
`N_FOLDS=5` is a development setting. Use `LOSO=True` for a final
leave-one-system-out evaluation; it trains one model per system and takes much
longer.

In [ ]:
SUITE = "all"              # datacenter, edge, or all
DIVISION = "closed"        # closed, open, or all
MODEL = "graphsage"        # graphsage, gcn, or gat
N_FOLDS = 5
LOSO = False
EPOCHS = 150
SEED = 7

experiment = long_table.copy()
if SUITE != "all":
    experiment = experiment.loc[experiment["suite"].eq(SUITE)].copy()
if DIVISION != "all":
    experiment = experiment.loc[experiment["division"].eq(DIVISION)].copy()
experiment = experiment.dropna(
    subset=["result", "log_performance", "system_id"]
).reset_index(drop=True)

print(f"Experiment rows: {len(experiment):,}")
print(f"Independent systems: {experiment['system_id'].nunique():,}")
display(
    experiment.groupby(["version", "division", "units"])
    .size()
    .rename("rows")
    .reset_index()
)

## 3. Inspect the heterogeneous GraphSAGE

There are five node types: `system`, `accelerator`, `processor`, `workload`,
and `result`. Typed edges express hardware containment, workload execution,
and observed measurements. The target is normalized within its exact
suite/version/division/model/task/scenario/unit group.

In [ ]:
model = gnn.HeteroGraphSAGE(
    hidden=64,
    layers=2,
    model_name=MODEL,
    dropout=0.2,
)

print(model)
print("\nTyped graph relations:")
for relation in gnn.graph_edge_types():
    print("  ", " -- ".join(relation))

## 4. Train GraphSAGE and the baselines

The notebook reports the target-group median, Ridge regression, random forest,
and GraphSAGE using the same system-held-out folds. Rows from one system are
never split between training and test.

In [ ]:
args = SimpleNamespace(
    data=canonical_path,
    suite=SUITE,
    folds=N_FOLDS,
    loso=LOSO,
    model=MODEL,
    hidden=64,
    layers=2,
    dropout=0.2,
    epochs=EPOCHS,
    patience=20,
    learning_rate=0.01,
    weight_decay=1e-4,
    seed=SEED,
    out_dir=Path("results") / "datacenter_edge",
)

summary, predictions = gnn.cross_validate(experiment, args)
group_report = gnn.grouped_report(predictions)

args.out_dir.mkdir(parents=True, exist_ok=True)
summary_path = args.out_dir / f"{SUITE}_{MODEL}_summary.csv"
prediction_path = args.out_dir / f"{SUITE}_{MODEL}_predictions.csv"
group_path = args.out_dir / f"{SUITE}_{MODEL}_groups.csv"
summary.to_csv(summary_path, index=False)
prediction_columns = [
    "fold", "method", "suite", "version", "division", "public_id",
    "organization", "system_id", "benchmark", "model", "task_code",
    "scenario", "units", "target_group", "result", "prediction", "y",
    "prediction_log",
]
predictions[prediction_columns].to_csv(prediction_path, index=False)
group_report.to_csv(group_path, index=False)

display(summary)

## 5. Inspect per-workload failures

Pooled MAPE can be dominated by a few extreme predictions. Always inspect the
per-target-group report before claiming that the model works broadly.

In [ ]:
selected = group_report.loc[group_report["method"].eq(MODEL)].copy()
selected = selected.sort_values("median_mape", ascending=False)
print(f"Highest {MODEL} median errors by target group:")
display(selected.head(12))

In [ ]:
plot_data = predictions.loc[
    predictions["method"].eq(MODEL)
    & predictions["result"].gt(0)
    & predictions["prediction"].gt(0)
]

fig, ax = plt.subplots(figsize=(7, 6))
ax.scatter(
    np.log10(plot_data["result"]),
    np.log10(plot_data["prediction"]),
    alpha=0.45,
    s=28,
    label=MODEL,
)
low = min(
    np.log10(plot_data["result"]).min(),
    np.log10(plot_data["prediction"]).min(),
)
high = max(
    np.log10(plot_data["result"]).max(),
    np.log10(plot_data["prediction"]).max(),
)
ax.plot([low, high], [low, high], color="black", linestyle="--", label="perfect")
ax.set_xlabel("log10 measured result (native units)")
ax.set_ylabel("log10 predicted result (native units)")
ax.set_title(f"{SUITE}: system-held-out {MODEL}")
ax.legend()
ax.grid(alpha=0.2)
plt.show()

In [ ]:
metric_plot = summary.set_index("method")["median_group_mape"].sort_values()
ax = metric_plot.plot.barh(figsize=(7, 3.5), color="#4c78a8")
ax.set_xlabel("Median MAPE (lower is better)")
ax.set_ylabel("")
ax.set_title("Baseline comparison by target group")
ax.grid(axis="x", alpha=0.2)
plt.tight_layout()
plt.show()

## 6. Saved outputs and interpretation

The summary, row-level predictions, and per-workload report are saved under
`results/datacenter_edge`. A strong GraphSAGE result should improve on the tabular
baselines across grouped folds, not merely in one aggregate metric.

For a system absent from training, an output remains an extrapolation until a
withheld measured result exists. These tables also do not identify utilization,
memory-bandwidth, or interconnect bottlenecks; those claims require profiler
telemetry and the roofline inputs listed in the project README.

In [ ]:
print("Summary:     ", summary_path.resolve())
print("Predictions: ", prediction_path.resolve())
print("Groups:      ", group_path.resolve())